In [2]:
import stripe
import os
import pandas as pd
from datetime import datetime, timedelta
import firebase_admin
# Firbease (pip install firebase_admin)
from firebase_admin import credentials, firestore
from google.cloud.firestore_v1.base_query import FieldFilter

import google.auth.transport.requests as grequests

from google.oauth2.id_token import fetch_id_token

In [78]:
OANACCOUNT = "5Tv2u4n8BReebmKUNIuN"

BANCOSTRIPESHOPIFY = "iA9Pzv2CImjItzwCaQv0"
BANCOSTRIPE = "oJMBTrVPmn2B4fBFZQ4e"
BANCOPAYPAL = "yfy4cPxkmFwIYVnoWD0A"
BANCOCAJAINGENIEROS = "vJbbj1kPxkcdXJyBOf1l"
DONACIONESACCOUNT = "I6vsoTqCKFAS1AK09qzW"
DONACIONESSOCIOSACCOUNT = "JTvjg76sbC4ZExoPVhCu"

ADMINGENERAL = "4zcptWXv2IqQFkIMz2MP"
ADMINGENERALINT2024 = "DrTngvUFYVLbF2oW8DLi"
ADMINGENERALPHASE2024 = "JeaSlPAPgLQOnYYAJdjl"
ESPAGNEGENERAL2024 = "FhnnK08Z47EkgPjZZ1NE"
NIKARITPROJECT = "0DmODGTOEiM5lg9SGx0J"

USERDANI = "z5m936GA0t3vHM28QKhR"
BOTUSER = "oXJJEfAEPxFYtdJ2pnaU"

CREATEDBYTYPESYSTEM = "system"

INCOMETYPE = "income"

SPREADSHEETID = os.environ["DIARIO_2024"]
creation_date = datetime.today().isoformat()[:-3] + "Z"

In [4]:
stripe.api_key = os.environ["STRIPE_KEY"]
cloudCreds = os.environ["CLOUD_CREDS"]

In [5]:
# Use a service account
cred = credentials.Certificate(cloudCreds)
firebase_admin.initialize_app(cred)

db = firestore.client()

In [91]:
##CREATE A DONATION DATA DICT
def createDonation(amount,accountingId, name, donor,exDate,i):
    randomId = db.collection('tmp').document().id
    
    data={
            "amount":amount,
            "accountingItem": accountingId,
            "code":i,
            "concept":"Donation from stripe "+name,
            "context":{
                "account":OANACCOUNT,
                "createdAt":creation_date,
                "createdBy":BOTUSER,
                "createdByType":CREATEDBYTYPESYSTEM,
                "id":randomId,
                "lastUpdateAt":creation_date,
                "lastUpdateBy":""
            },
            "description":None,
            "donor": donor,
            "executedAt":exDate,
            "intervention":ADMINGENERALINT2024, 
            "project": ADMINGENERAL
        }
    
    donationItem = {
                    "data":{
                            "donation": data
                            }
                    }
    return donationItem


In [79]:
def createAccountingDonation(amount, name, banco, date, i):
    randomId = db.collection("tmp").document().id
    data = {
        "amount": float(amount),
        "baseAmount": float(amount),
        "code": i,
        "concept": "Donación OF " + name + ' ' + date[0:10],
        "context": {
            "account": OANACCOUNT,
            "createdAt": creation_date,
            "createdBy": BOTUSER,
            "createdByType": CREATEDBYTYPESYSTEM,
            "id": randomId,
            "lastUpdateAt": creation_date,
            "lastUpdateBy": "",
        },
        "description": None,
        "executedAt": date,
        "files": None,
        "originAccountingAccount": DONACIONESSOCIOSACCOUNT,
        "originIntervention": None,
        "originPhase": None,
        "originProject": None,
        "responsible": USERDANI,
        "tags": ["spain"],
        "targetAccountingAccount": banco,
        "targetIntervention": ADMINGENERALINT2024,
        "targetPhase": ADMINGENERALPHASE2024,
        "targetProject": ADMINGENERAL,
        "type": INCOMETYPE,
        "vat": 0,
        "vatAmount": 0,
    }
    accountItem = {"data": {"accountingItem": data}}
    return accountItem


In [9]:
customers = stripe.Customer.list()

In [10]:
customer_data = [
    {
        "idStripe": customer.id,
        "email": customer.email,
        "created": customer.created,
        "metadata": customer.metadata
    }
    for customer in customers.auto_paging_iter()
]

df_customers = pd.DataFrame(customer_data)


In [11]:
df_customers['created'] = pd.to_datetime(df_customers['created'], unit='s')

In [12]:
## COGER TODAS LOS DONANTES DE FIREBASE
donorsFirebase = (
    db.collection("donors")
    .where(filter=FieldFilter("context.account", "==", OANACCOUNT))
    .stream()
)
l_donors = []
for donorD in donorsFirebase:
    # print(payout)
    l_donors.append(donorD.to_dict())

In [13]:
df_donors = pd.DataFrame(l_donors)

In [14]:
df_donors = df_donors.join(df_donors["context"].apply(pd.Series))
df_recurrentConfig = df_donors["recurrentConfig"].apply(pd.Series)
df_recurrentConfig = df_recurrentConfig.drop(columns=[0])
df_donors = df_donors.join(df_recurrentConfig)

In [15]:
df_donors = df_donors.drop(columns=["context"])
df_donors = df_donors.drop(columns=["recurrentConfig"])
df_donors = df_donors.drop(columns=["customFields"])

In [16]:
df_customers_copy_stripe = df_customers.copy()

In [17]:
#df_customers = df_customers_copy_stripe.copy()

In [18]:
df_customers = pd.merge(df_customers,df_donors[['id','email','name','lastName']], on="email",how="left")

In [19]:
df_customers['name'] = df_customers['name'] + ' ' + df_customers['lastName']

In [20]:
df_customers = df_customers.drop(columns=["lastName"])

In [21]:
# Obtener el timestamp UNIX del inicio del último año
one_year_ago = datetime.now() - timedelta(days=365)
timestamp_one_year_ago = int(one_year_ago.timestamp())

charges = stripe.Charge.list(
    created={'gte': timestamp_one_year_ago}
)

In [51]:
charge_data = [
    {
        "charge_id": charge.id,
        "amount": charge.amount / 100,  # Convertir de centavos a la unidad monetaria
        "currency": charge.currency,
        "created": datetime.fromtimestamp(charge.created).isoformat(),
        "customer": charge.customer,
        "description": charge.description,
        "payment_method_type": charge.payment_method_details.type
    }
    for charge in charges.auto_paging_iter() if charge.status == 'succeeded'
]

In [52]:
df_charges = pd.DataFrame(charge_data)

In [53]:
df_charges_copy_stripe = df_charges.copy()

In [54]:
df_charges = pd.merge(df_charges, df_customers[['idStripe','id','email','name']],left_on = "customer",right_on="idStripe",how='left')

In [87]:
df_charges = df_charges[~df_charges['email'].isna()]

In [103]:
url_onCreateDonationAPI = os.environ['ONCREATE_DONATION_URL']
auth_req = grequests.Request()
id_token = fetch_id_token(auth_req, url_onCreateDonationAPI)
headers_donations = {
    "Authorization": "Bearer {}".format(id_token),
    "Content-type": "application/json",
}
url_onCreateAccountingItemAPI = os.environ['ONCREATE_ACCOUNTINGITEM_URL']
auth_req = grequests.Request()
id_token = fetch_id_token(auth_req, url_onCreateAccountingItemAPI)
headers_accounting = {
    "Authorization": "Bearer {}".format(id_token),
    "Content-type": "application/json",
}

In [104]:
df_charges2024 = df_charges[(df_charges['created']<'2025')]

In [105]:
df_charges2024 = df_charges2024.sort_values(by='created')
d_charges2024 = df_charges2024.T.to_dict()

In [123]:
counter_accountingItems = (
    db.collection("info").document(OANACCOUNT + "-accountingItems").get().to_dict()
)
counter_donations = (
    db.collection("info").document(OANACCOUNT + "-donations").get().to_dict()
)

i_donations = counter_donations['counter']
i_accounting = counter_accountingItems['counter']
donationItems = []
accountingItems = []

for chargeid in d_charges2024:
    # CHECK IF IT IS ALREADY MATCHED
    ##get row in Caja de Ingenieros
    charge = d_charges2024[chargeid]
    amount = charge['amount']
    name = charge['name']
    date = charge['created']
    payment_method = charge['payment_method_type']
    donor = charge['id']
    if payment_method == "paypal":
        banco = BANCOPAYPAL
    else:
        banco = BANCOSTRIPE
    accountingItem = createAccountingDonation(amount,name,banco,date,i_accounting)
    id_accounting = accountingItem['data']['accountingItem']['context']['id']
    donationItem = createDonation(amount,id_accounting,name,donor,date,i_donations)
    i_donations = i_donations+1
    i_accounting = i_accounting+1
    r = grequests.requests.post(url_onCreateAccountingItemAPI, json=accountingItem, headers=headers_accounting)
    if r.status_code >204:
        print(r.status_code,"error in accounting " + name + ' / '+date)
    r = grequests.requests.post(url_onCreateDonationAPI, json=donationItem, headers=headers_donations)
    if r.status_code >204:
        print(r.status_code,"error in donation" + name + ' / '+date)
    accountingItems.append(accountingItem['data']['accountingItem'])
    donationItems.append(accountingItem['data']['accountingItem'])

db.collection("info").document(OANACCOUNT + "-accountingItems").update({"counter": i_accounting})
db.collection("info").document(OANACCOUNT + "-donations").update({"counter": i_donations})

KeyboardInterrupt: 

In [107]:
df_accountingItems = pd.DataFrame(accountingItems)
df_accountingItems = df_accountingItems.join(df_accountingItems["context"].apply(pd.Series))
df_accountingItems = df_accountingItems.drop(columns=["context"])
df_donationItems = pd.DataFrame(donationItems)
df_donationItems = df_donationItems.join(df_donationItems["context"].apply(pd.Series))
df_donationItems = df_donationItems.drop(columns=["context"])
df_accountingItems.to_csv("accountingdonations.csv")
df_donationItems.to_csv("donations.csv")